# Практикум 1. Приёмка данных и базовая линия аномалий (Google Colab)
**После этого блокнота вы умеете:** (1) принять выгрузку журнала проводок и вынести вердикт «принять / вернуть» по набору проверок; (2) измерить дрейф между периодами и найти сегмент, где он возник; (3) собрать базовую линию (правила → Isolation Forest) и честно сравнить методы по precision@k; (4) разобрать журнал событий в шаблоны.
**Данные:** синтетическая главная книга с известными дефектами и закладками (ground truth) и открытый журнал Loghub HDFS_2k. Внутренние данные в Colab не загружаем.
**Артефакты на выходе:** `data_passport.csv`, `drift_report.csv`, `baseline_metrics.json`, `handoff.json` (читает блокнот П2). Запуск: Среда выполнения → Выполнить всё.

In [ ]:
import numpy as np, pandas as pd, json, re, hashlib, urllib.request
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
print('pandas', pd.__version__)
SEED = 42; rng = np.random.default_rng(SEED)

## 0. Данные: два квартала с заложенными дефектами
`osv` — контрольные итоги по счетам (как оборотно-сальдовая ведомость), посчитаны **до** потери строк. Это то, что в реальной жизни вы получаете отдельно от выгрузки.

In [ ]:
ACC = ['4010','4020','5010','5020','6010','7010']; HR = {f'u{i:02d}' for i in range(30)}
def make_period(n, start, mu, id0, rate=0.012):
    d = pd.DataFrame({'entry_id': np.arange(n)+id0})
    day = rng.integers(0, 90, n); h = rng.choice(24, n, p=np.r_[np.full(7,.002),np.full(11,.085),np.full(6,.0095)]/(.002*7+.085*11+.0095*6))
    d['created_ts'] = pd.Timestamp(start)+pd.to_timedelta(day,'D')+pd.to_timedelta(h*3600+rng.integers(0,3600,n),'s')
    d['post_date'] = d.created_ts.dt.normalize()+pd.to_timedelta(rng.poisson(1.2,n),'D')
    d['amount'] = np.round(rng.lognormal(mu,1.1,n),2)
    d['account'] = rng.choice(ACC,n,p=[.3,.2,.2,.15,.1,.05]); d['cost_center'] = rng.choice(['CC01','CC02','CC03','CC04','CC05'],n)
    d['user'] = rng.choice(sorted(HR),n); d['doc_type'] = rng.choice(['INV','PAY','ADJ'],n,p=[.6,.3,.1]); d['is_anomaly'] = 0
    idx = rng.choice(n,int(n*rate),replace=False); a,b,c = np.array_split(idx,3)
    d.loc[a,'amount'] = rng.choice([50000,100000,250000],len(a))
    d.loc[b,'amount'] = np.round(rng.uniform(9500,9999.99,len(b)),2)
    d.loc[c,'user'] = 'u_rare'; d.loc[c,'created_ts'] = d.loc[c,'created_ts'].dt.normalize()+pd.Timedelta(hours=2,minutes=17)
    d.loc[idx,'is_anomaly'] = 1
    osv = d.groupby('account').amount.sum().round(2)                       # контрольные итоги до дефектов
    d = d.drop(rng.choice(n,int(.003*n),replace=False)).reset_index(drop=True)   # потерянные при выгрузке строки
    m = len(d); d.loc[rng.choice(m,int(.02*m),replace=False),'cost_center'] = np.nan
    d.loc[rng.choice(m,int(.005*m),replace=False),'account'] = '9999'
    d.loc[rng.choice(m,int(.003*m),replace=False),'amount'] = 0.0
    j = rng.choice(m,int(.002*m),replace=False); d.loc[j,'post_date'] = d.loc[j,'created_ts'].dt.normalize()-pd.Timedelta(days=1)
    j = rng.choice(m,int(.01*m),replace=False); d.loc[j,'post_date'] += pd.Timedelta(days=20)
    nd = d.sample(int(.003*m),random_state=SEED).copy(); nd['entry_id'] = np.arange(len(nd))+id0+n; nd['created_ts'] += pd.Timedelta(days=1); nd['is_anomaly'] = 0
    ex = d.sample(int(.004*m),random_state=SEED+1)
    return pd.concat([d,nd,ex],ignore_index=True), osv
q1, osv1 = make_period(20000,'2026-01-01',7.0,0)
q2, osv2 = make_period(20000,'2026-04-01',7.25,10**6)
print(q1.shape, q2.shape, 'доля закладок', q2.is_anomaly.mean().round(4))

## 1. Приёмка выгрузки
Каждая проверка возвращает **число записей-нарушителей**. Вердикт строится по порогам, а не по «среднему баллу».

In [ ]:
def fingerprint(df):                               # 1. отпечаток выгрузки: фиксирует версию данных
    return hashlib.sha256(pd.util.hash_pandas_object(df.drop(columns='is_anomaly'),index=False).values.tobytes()).hexdigest()[:16]

def check_nulls(df, mandatory=('amount','account','cost_center','user','post_date')):
    return df[list(mandatory)].isna().mean().max()                                  # худшее поле, доля

def check_dups(df):
    exact = df.drop(columns='is_anomaly').duplicated().sum()                          # полные дубли строк
    k = ['user','account','amount']; s = df.drop_duplicates().sort_values(k+['created_ts'])
    near = (s.groupby(k).created_ts.diff() < pd.Timedelta(days=3)).sum()               # та же сумма/счёт/автор в окне 3 дня
    return exact, near

def check_gaps(df):                                # 2. полнота совокупности: пропуски в нумерации
    ids = np.sort(df.entry_id.unique()); return (ids.max()-ids.min()+1) - len(ids)

def check_recon(df, osv):                          # 3. сверка итогов по счетам с контрольными
    fact = df.drop_duplicates().query("account in @ACC").groupby('account').amount.sum()
    return ((fact-osv).abs()/osv).max(), ((osv-fact).sum())

def benford_mad(x):                                # 4. закон Бенфорда: средн. абс. отклонение первой цифры
    x = x[x>0]; fd = (x/10**np.floor(np.log10(x))).astype(int)
    p = fd.value_counts(normalize=True).reindex(range(1,10),fill_value=0).values
    return float(np.abs(p-np.log10(1+1/np.arange(1,10))).mean())

In [ ]:
def passport(df, osv, qend, raw=None):
    raw = df if raw is None else raw                       # полноту и сверку считаем по сырой выгрузке
    n = len(df); ex, near = check_dups(df); gaps = check_gaps(raw); rdiff, rsum = check_recon(raw, osv)
    lag = (df.post_date-df.created_ts.dt.normalize()).dt.days
    rows = [('Сверка с ОСВ: макс. расхождение по счёту', rdiff, .005, True),
            ('Разрывы нумерации (доля)', gaps/len(raw), .005, True),
            ('Полные дубли', ex/n, .005, True),
            ('Обязательные поля: худшая доля пропусков', check_nulls(df), .01, True),
            ('Счёт не из плана счетов', (~df.account.isin(ACC)).mean(), .003, True),
            ('Проводка раньше создания', (lag<0).mean(), .001, True),
            ('Почти-дубли (окно 3 дня)', near/n, .005, False),
            ('Автор не из справочника HR', (~df.user.isin(HR)).mean(), .001, False),
            ('Сумма <= 0', (df.amount<=0).mean(), .001, False),
            ('Задержка проводки > 5 дней', (lag>5).mean(), .02, False),
            ('Сдвиг периода (cutoff)', ((df.created_ts.dt.normalize()>qend-pd.Timedelta(days=3))&(df.post_date>qend)).mean(), .01, False)]
    P = pd.DataFrame(rows, columns=['check','value','threshold','blocker']); P['fail'] = P.value>P.threshold
    P['status'] = np.where(P.fail, np.where(P.blocker,'БЛОКЕР','ВНИМАНИЕ'), 'ок')
    return P, ('ВЕРНУТЬ ВЛАДЕЛЬЦУ' if (P.fail&P.blocker).any() else 'ПРИНЯТЬ'), rsum
P1, verdict, lost = passport(q1, osv1, pd.Timestamp('2026-03-31'))
print('Отпечаток', fingerprint(q1), '| Вердикт:', verdict, '| недостаёт по сумме:', f'{lost:,.0f}'); print(P1[['check','value','threshold','status']])
print('Бенфорд MAD:', round(benford_mad(q1.amount),4), '(<0.006 близко, >0.015 не соответствует)'); P1.to_csv('data_passport.csv', index=False)

## 2. Очистка с журналом исключений
Мы **не чиним** то, что можно вернуть владельцу (потерянные строки), но убираем то, что безопасно исключить, и фиксируем сколько и почему.

In [ ]:
def clean(df):
    log = {}; n0 = len(df)
    df = df.drop_duplicates(subset=[c for c in df.columns if c!='is_anomaly']); log['полные дубли'] = n0-len(df)
    bad = ~df.account.isin(ACC)|(df.amount<=0); log['счёт/сумма невалидны'] = int(bad.sum()); df = df[~bad]
    lag = (df.post_date-df.created_ts.dt.normalize()).dt.days; log['проводка раньше создания'] = int((lag<0).sum()); df = df[lag>=0]
    return df.reset_index(drop=True), log
q1c, log1 = clean(q1); q2c, log2 = clean(q2); print(log2, '→ строк', len(q2c))
P2, v2, _ = passport(q2c, osv2, pd.Timestamp('2026-06-30'), raw=q2); print(P2[P2.fail][['check','value','status']])

## 3. Дрейф: PSI по признаку и по сегментам
`PSI = Σ (p − q)·ln(p / q)`; бины — квантили **эталона**. Ориентиры: <0.10 стабильно, 0.10–0.25 внимание, ≥0.25 дрейф. Смотрим на размер эффекта, а не на p-value (на 20 000 строк KS «значим» почти всегда).

In [ ]:
def psi(ref, cur, bins=10, eps=1e-6):
    e = np.unique(np.quantile(ref, np.linspace(0,1,bins+1))); e[0], e[-1] = -np.inf, np.inf
    p = np.clip(np.histogram(ref,e)[0]/len(ref), eps, None); q = np.clip(np.histogram(cur,e)[0]/len(cur), eps, None)
    return float(np.sum((p-q)*np.log(p/q)))
ks = stats.ks_2samp(q1c.amount, q2c.amount)
print('amount: PSI', round(psi(q1c.amount,q2c.amount),3), '| KS', round(ks.statistic,3), '| p', ks.pvalue)
seg = pd.DataFrame({a: {'PSI': psi(q1c[q1c.account==a].amount, q2c[q2c.account==a].amount), 'n_ref': (q1c.account==a).sum()} for a in ACC}).T
seg['status'] = pd.cut(seg.PSI,[-1,.1,.25,9],labels=['ок','внимание','дрейф']); print(seg); seg.to_csv('drift_report.csv')

## 4. Признаки, правила и Isolation Forest
Модель **обучаем на Q1, оцениваем на Q2** (сплит по времени). Правила — базовая линия, которую ML обязан побить.

In [ ]:
def feats(d, ref):
    f = pd.DataFrame(index=d.index); f['log_amt'] = np.log1p(d.amount); f['hour'] = d.created_ts.dt.hour
    f['night'] = ((f.hour<6)|(f.hour>=22)).astype(int); f['weekend'] = (d.created_ts.dt.dayofweek>=5).astype(int)
    f['round'] = ((d.amount%1000==0)&(d.amount>=10000)).astype(int); f['near_limit'] = d.amount.between(9500,9999.99).astype(int)
    f['user_freq'] = d.user.map(ref.user.value_counts(normalize=True)).fillna(0)       # частоты берём из эталона
    f['lag'] = (d.post_date-d.created_ts.dt.normalize()).dt.days; return f
F1, F2 = feats(q1c,q1c), feats(q2c,q1c); y = q2c.is_anomaly.values
rules = pd.DataFrame({'круглая крупная': F2['round'], 'ниже лимита 10 000': F2.near_limit, 'ночь + редкий автор': ((F2.night==1)&(F2.user_freq<.01)).astype(int)})
rt = pd.DataFrame({'сработало': rules.sum(), 'precision': [y[rules[c]==1].mean() for c in rules]}); print(rt)

In [ ]:
iso = IsolationForest(n_estimators=300, max_samples=256, random_state=SEED, n_jobs=-1).fit(F1)   # contamination не задаём: ранжируем
s_iso = -iso.score_samples(F2)
med = np.median(F1.log_amt); mad = np.median(np.abs(F1.log_amt-med)); s_rz = np.abs(0.6745*(F2.log_amt-med)/mad).values
s_rule = rules.sum(axis=1).values + 1e-3*s_rz                                        # правила, ничьи разрешаем суммой
pk = lambda s,k: float(y[np.argsort(-s)[:k]].mean())
K = [50,200,500]; R = {m:{f'P@{k}':pk(s,k) for k in K} for m,s in [('правила',s_rule),('Isolation Forest',s_iso),('робастный z',s_rz)]}
for m,s in [('правила',s_rule),('Isolation Forest',s_iso),('робастный z',s_rz)]: R[m]['PR-AUC'] = float(average_precision_score(y,s))
R['случайный выбор'] = {f'P@{k}': float(y.mean()) for k in K}; R['доля аномалий'] = float(y.mean())
print(pd.DataFrame(R).T.round(3)); json.dump(R, open('baseline_metrics.json','w'), indent=1, ensure_ascii=False)

## 5. Журнал событий: шаблоны сообщений (Loghub HDFS_2k)
Источник: Zhu et al., *Loghub* (arXiv:2008.06448), GitHub `logpai/loghub`. Нормализация переменных частей превращает 2000 строк в десятки шаблонов — на них считают частоты и редкие события.

In [ ]:
URL = 'https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log'
try:
    raw = urllib.request.urlopen(URL, timeout=20).read().decode().splitlines()
    rx = re.compile(r'^(\d{6}) (\d{6}) (\d+) (\w+) ([\w.$]+): (.*)$'); L = [m.groups() for m in map(rx.match, raw) if m]
    log = pd.DataFrame(L, columns=['date','time','pid','level','component','msg'])
    log['tpl'] = (log.msg.str.replace(r'blk_-?\d+','<BLK>',regex=True).str.replace(r'\d+\.\d+\.\d+\.\d+(:\d+)?','<IP>',regex=True).str.replace(r'\d+','<N>',regex=True))
    print('разобрано', len(log), 'из', len(raw), '| шаблонов', log.tpl.nunique()); print(log.level.value_counts().to_dict())
    print(log.tpl.value_counts().tail(3))       # редкие шаблоны — первые кандидаты на разбор
except Exception as e: print('Loghub недоступен:', e)

In [ ]:
handoff = {'fingerprint_q1': fingerprint(q1), 'fingerprint_q2': fingerprint(q2), 'verdict_q1_raw': verdict, 'excluded_q2': {k:int(v) for k,v in log2.items()},
           'artifacts': ['data_passport.csv','drift_report.csv','baseline_metrics.json']}
json.dump(handoff, open('handoff.json','w'), indent=1, ensure_ascii=False); print(handoff)

## Задания
1. Подберите порог блокера для разрывов нумерации так, чтобы гейт различал «потеряно 0,3%» и «потеряно 3%» — проверьте на двух значениях `.003`.
2. Добавьте проверку «круглые суммы» в `passport()` и решите: блокер или внимание.
3. Внесите дрейф в один счёт Q2 и найдите его в `seg`.
4. Сравните P@k при k=50/200/500 и выберите k под ёмкость проверки 100 проводок в неделю.
5. Найдите в Loghub три самых редких шаблона и объясните, что делать с ними аудитору.